<a href="https://colab.research.google.com/github/KadleczPaul/SQL-fundamentals-DataCamp/blob/main/E_commerce_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install duckdb

import pandas as pd
import duckdb

con = duckdb.connect(database=':memory:')

data = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'user_id': [101, 102, 101, 103, 102, 101, 104, 105, 102, 103],
    'order_date': pd.to_datetime([
        '2023-01-15', '2023-01-20', '2023-02-05', '2023-02-12', '2023-03-01',
        '2023-03-15', '2023-03-20', '2023-04-02', '2023-04-10', '2023-04-25'
    ]),
    'amount': [150, 200, 50, 300, 100, 400, 250, 80, 120, 600],
    'category': ['Tech', 'Home', 'Tech', 'Fashion', 'Home', 'Tech', 'Fashion', 'Books', 'Home', 'Tech']
})


con.register('orders', data)
print("Baza de date a fost creată și e gata pentru SQL!")

Baza de date a fost creată și e gata pentru SQL!


In [40]:
query = """
  select user_id,
    count(order_id) as numar_comenzi,
    sum(amount) as suma_totala,
    min(order_date) as comanda_veche,
    max(order_date) as comanda_recenta
from orders
group by user_id
having count(order_id) > 1
order by sum(amount) DESC
"""

con.execute(query).df()

,user_id,numar_comenzi,suma_totala,comanda_veche,comanda_recenta
0,103,2,900.0,2023-02-12,2023-04-25
1,101,3,600.0,2023-01-15,2023-03-15
2,102,3,420.0,2023-01-20,2023-04-10


In [41]:
query2 = """
SELECT category,
  count(order_id) AS numar_comenzi,
  sum(order_id) AS suma_totala,
  avg(amount) AS valoare_medie
FROM orders
WHERE amount > 100
group by category
order by category ASC;

"""
con.execute(query2).df()

,category,numar_comenzi,suma_totala,valoare_medie
0,Fashion,2,11.0,275.000000
1,Home,2,11.0,160.000000
2,Tech,3,17.0,383.333333


In [42]:
query3 = """
WITH premium AS (
    SELECT AVG(amount) AS medie_totala
    FROM orders
)
SELECT order_id,
       user_id,
       category,
       amount
FROM orders
WHERE amount > (SELECT medie_totala FROM premium)
ORDER BY amount DESC;
"""

con.execute(query3).df()

,order_id,user_id,category,amount
0,10,103,Tech,600
1,6,101,Tech,400
2,4,103,Fashion,300
3,7,104,Fashion,250
